# Sales Data Cleaning & Visualization

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import missingno as msno
pd.set_option('display.max_columns',200)
pd.set_option('display.width',200)
np.random.seed(42)
sns.set(style='whitegrid')

In [ ]:
DATA_PATH = '../data/sales_data_sample.csv'
df = pd.read_csv(DATA_PATH)
print(DATA_PATH, df.shape)
df.head()

In [ ]:
df.info(memory_usage='deep')
print(df.shape)
print(df.isnull().sum().to_dict())
print(df.nunique().to_dict())
display(df.describe(include='all').T)

In [ ]:
def impute_missing(df,strategy='median'):
    df2 = df.copy()
    num_cols = df2.select_dtypes(include=[np.number]).columns
    cat_cols = df2.select_dtypes(include=['object','category']).columns
    if strategy == 'drop':
        return df2.dropna()
    if strategy == 'mean':
        df2[num_cols] = df2[num_cols].fillna(df2[num_cols].mean())
    elif strategy == 'median':
        df2[num_cols] = df2[num_cols].fillna(df2[num_cols].median())
    elif strategy == 'ffill':
        df2 = df2.fillna(method='ffill').fillna(method='bfill')
    for c in cat_cols:
        if df2[c].isnull().any():
            modes = df2[c].mode()
            df2[c] = df2[c].fillna(modes.iloc[0] if not modes.empty else '')
    return df2
df = impute_missing(df,'median')

In [ ]:
print(int(df.duplicated().sum()))
df = df.drop_duplicates()
print(int(df.duplicated().sum()))

In [ ]:
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month

In [ ]:
from scipy import stats
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if num_cols:
    Q1 = df[num_cols].quantile(0.25)
    Q3 = df[num_cols].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    for c in num_cols:
        df[c] = df[c].clip(lower=lower[c], upper=upper[c])

In [ ]:
if {'unit_price','quantity'}.issubset(df.columns):
    df['total_sales'] = df['unit_price'] * df['quantity']
elif {'price','qty'}.issubset(df.columns):
    df['total_sales'] = df['price'] * df['qty']

In [ ]:
from sklearn.preprocessing import StandardScaler
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if num_cols:
    scaler = StandardScaler()
    df_scaled = df.copy()
    df_scaled[num_cols] = scaler.fit_transform(df_scaled[num_cols].fillna(0))

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    plt.figure(figsize=(6,3))
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(col)
    plt.show()
cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
for c in cat_cols:
    plt.figure(figsize=(6,3))
    sns.countplot(y=c, data=df, order=df[c].value_counts().index[:10])
    plt.title(c)
    plt.show()

In [ ]:
if len(num_cols) >= 2:
    sns.pairplot(df[num_cols].dropna().sample(min(200, len(df))))
    plt.show()
if len(num_cols) >= 2:
    plt.figure(figsize=(8,6))
    sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
    plt.show()

Run app.py for an interactive Streamlit dashboard

In [ ]:
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
output_dir = f'../outputs/run_{ts}'
os.makedirs(output_dir, exist_ok=True)
df.to_csv(os.path.join(output_dir, 'cleaned_sales.csv'), index=False)
if 'total_sales' in df.columns:
    plt.figure(figsize=(6,3))
    sns.histplot(df['total_sales'].dropna(), kde=True)
    plt.title('total_sales')
    plt.savefig(os.path.join(output_dir, 'total_sales_hist.png'))
    plt.close()
print('Artifacts written to', output_dir)

In [ ]:
import pkg_resources
for p in ['pandas','numpy','matplotlib','seaborn','plotly','missingno','scipy','sklearn']:
    try:
        print(p, pkg_resources.get_distribution(p).version)
    except Exception:
        print(p, 'not installed')
print('Notebook updated for sales data.')